# Kayıp Fonksiyonları

Bu alıştırmada, Kayıp fonksiyonlarının `LinearRegression` modeli üzerindeki etkilerini karşılaştıracaksınız.

👇 Bu zorluk için kullanmak üzere bir CSV dosyası indirelim ve onu bir DataFrame'e dönüştürelim

In [2]:
import pandas as pd

data = pd.read_csv("https://d32aokrjazspmn.cloudfront.net/materials/loss_functions_dataset.csv")
data.sample(5)

,Relative Compactness,Surface Area,Wall Area,Roof Area,Overall Height,Glazing Area,Average Temperature
591,0.82,612.5,318.5,147.0,7.0,0.40,30.465
91,0.64,784.0,343.0,220.5,3.5,0.10,17.265
112,0.79,637.0,343.0,147.0,7.0,0.10,38.360
131,0.69,735.0,294.0,220.5,3.5,0.10,12.500
497,0.79,637.0,343.0,147.0,7.0,0.25,38.245


🎯 Göreviniz, tasarımına göre bir seranın içindeki ortalama sıcaklığı tahmin etmektir. Sıcaklık tahminleriniz, her bir bitki için iklim ihtiyaçlarına göre uygun sera tasarımını seçmenize yardımcı olacaktır.

🌿 Bitkilerin küçük sıcaklık değişimlerini kaldırabildiğini, ancak sıcaklık değişimleri arttıkça katlanarak daha duyarlı hale geldiğini biliyorsunuz.

## 1. Teori

❓ Teorik olarak, bitkileri öldürme riskini sınırlamak için modelinizi hangi Kayıp fonksiyonu üzerinde eğitirsiniz?

<details>
<summary> 🆘 Cevap </summary>
    
Teorik olarak, Ortalama Kare Hata (MSE) Kayıp fonksiyonunu kullanırsınız. Bu, aykırı tahminleri cezalandırır ve modelinizin büyük hatalar yapmasını engeller. Bu, daha küçük sıcaklık değişimleri ve bitkiler için daha düşük risk sağlayacaktır.

</details>

> CEVABINIZI BURAYA YAZIN

## 2. Uygulama

### 2.1 Ön İşleme

❓ Özellikleri standartlaştırın

In [7]:
from sklearn.preprocessing import StandardScaler

X = data.loc[:,'Relative Compactness':'Glazing Area']  

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


### 2.2 Modelleme

Bu bölümde, farklı Kayıp fonksiyonları üzerinde optimize edilmiş modelleri değerlendirerek teoriyi doğrulayacaksınız.

### En Küçük Kareler (MSE) Kaybı

❓ **En Küçük Kareler Kaybı** (MSE) üzerinde **Stokastik Gradyan İnişi** (SGD) ile optimize edilmiş bir Doğrusal Regresyon modelini **10-Katlı Çapraz doğrula**

In [8]:
import numpy as np 
from sklearn.model_selection import cross_validate
from sklearn.linear_model import SGDRegressor
sgd_model = SGDRegressor(loss = "squared_error")

sgd_model_cv = cross_validate(
    sgd_model, 
    X_scaled, 
    data['Average Temperature'],
    cv = 10, 
    scoring = ['r2','max_error']
)

sgd_model_cv
                         

{'fit_time': array([0.0094049 , 0.00385785, 0.00374007, 0.00307584, 0.00287485,
        0.00334311, 0.00199509, 0.00250196, 0.00257516, 0.00262618]),
 'score_time': array([0.00159621, 0.00067902, 0.00076795, 0.00057817, 0.00082302,
        0.00061679, 0.00054121, 0.00048494, 0.00045896, 0.000458  ]),
 'test_r2': array([0.78645703, 0.90924501, 0.89566196, 0.88363954, 0.93109799,
        0.89685592, 0.92729228, 0.91650962, 0.89574934, 0.93942613]),
 'test_max_error': array([-9.87579713, -8.68403266, -8.81119296, -9.28544837, -8.98888143,
        -8.53611697, -8.58588317, -8.72518293, -8.33288509, -7.61121651])}

❓ Hesaplayın:
- Ortalama çapraz doğrulanmış R2 skoru ve bunu `r2` değişkeninde kaydedin
- Tüm katlarınızın °C cinsinden en büyük tek tahmin hatasını hesaplayın ve `max_error_celsius` değişkeninde kaydedin

(İpucu: `max_error` sklearn'de kabul edilen bir puanlama metriğidir)

In [9]:
r2 = sgd_model_cv['test_r2'].mean()
r2

0.8981934809715358

In [11]:
max_error_celsius = abs(sgd_model_cv['test_max_error']).max()
max_error_celsius

9.875797130771495

### Ortalama Mutlak Hata (MAE) Kaybı

Peki modelimizi MAE üzerinde optimize edersek ne olur?

❓ **MAE** Kaybı üzerinde **Stokastik Gradyan İnişi** (SGD) ile optimize edilmiş bir Doğrusal Regresyon modelini **10-Katlı Çapraz doğrula**

<details>
<summary>💡 İpuçları</summary>

- MAE kaybı `SGDRegressor`'da doğrudan belirtilemez. Doğru parametreleri ayarlayarak tasarlanması gerekir

</details>

In [12]:
mae_model= SGDRegressor(loss="epsilon_insensitive", epsilon = 0)

mae_model_cv = cross_validate(
    mae_model, 
    X_scaled, 
    data['Average Temperature'],
    cv = 10, 
    scoring = ['r2','max_error']
)

sgd_model_cv

{'fit_time': array([0.0094049 , 0.00385785, 0.00374007, 0.00307584, 0.00287485,
        0.00334311, 0.00199509, 0.00250196, 0.00257516, 0.00262618]),
 'score_time': array([0.00159621, 0.00067902, 0.00076795, 0.00057817, 0.00082302,
        0.00061679, 0.00054121, 0.00048494, 0.00045896, 0.000458  ]),
 'test_r2': array([0.78645703, 0.90924501, 0.89566196, 0.88363954, 0.93109799,
        0.89685592, 0.92729228, 0.91650962, 0.89574934, 0.93942613]),
 'test_max_error': array([-9.87579713, -8.68403266, -8.81119296, -9.28544837, -8.98888143,
        -8.53611697, -8.58588317, -8.72518293, -8.33288509, -7.61121651])}

❓ Hesaplayın:
- Ortalama çapraz doğrulanmış R2 skoru, bunu `r2_mae`'de saklayın
- Tüm katlarınızın en büyük tek tahmin hatasını, bunu `max_error_mae`'de saklayın

In [25]:
r2_mae = sgd_model_cv['test_r2'].mean()
r2_mae

0.8981934809715358

In [26]:
max_error_mae = abs(mae_model_cv['test_max_error']).max()
max_error_mae

11.183907922634312

## 3. Sonuç

❓ Değerlendirdiğiniz modellerden hangisi göreviniz için en uygun görünüyor?

<details>
<summary> 🆘Cevap </summary>
    
İki model arasında ortalama çapraz doğrulanmış r2 skorları yaklaşık olarak benzer olmasına rağmen, MAE üzerinde optimize edilen modelin zaman zaman daha büyük hatalar yapma şansı daha fazladır, bu da bitkileri öldürme riskini artırır!
    
</details>

> CEVABINIZI BURAYA YAZIN

# 🏁 Kodunuzu kontrol edin ve notebook'unuzu gönderin

In [27]:
from nbresult import ChallengeResult

result = ChallengeResult(
    'loss_functions',
    r2 = r2,
    r2_mae = r2_mae,
    max_error = max_error_celsius,
    max_error_mae = max_error_mae
)

result.write()
print(result.check())


============================= test session starts ==============================
platform darwin -- Python 3.12.9, pytest-8.3.4, pluggy-1.5.0 -- /Users/iremtaskin/.pyenv/versions/3.12.9/envs/workintech/bin/python
cachedir: .pytest_cache
rootdir: /Users/iremtaskin/Workintech/14-week/S16D4-S-loss-functions/tests
plugins: anyio-4.8.0, typeguard-4.4.2
collecting ... collected 3 items

test_loss_functions.py::TestLossFunctions::test_max_error_order PASSED   [ 33%]
test_loss_functions.py::TestLossFunctions::test_r2 PASSED                [ 66%]
test_loss_functions.py::TestLossFunctions::test_r2_mae PASSED            [100%]

============================== 3 passed in 0.05s ===============================


💯 You can commit your code:

git add tests/loss_functions.pickle

git commit -m 'Completed loss_functions step'

git push origin master

